# 03 — Preprocessing and Split


In [ ]:
BRANCH = "main"

%cd /kaggle/working
!rm -rf damage-detection
!git clone -b {BRANCH} https://github.com/azrairemd/damage-detection.git
%cd damage-detection
!pip install -q -r requirements.txt
!pip install -q -e .

/kaggle/working
Cloning into 'damage-detection'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 164 (delta 64), reused 129 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 11.33 MiB | 37.68 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/kaggle/working/damage-detection
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for damage-detection (pyproject.toml) ... done


In [19]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from collections import Counter
from PIL import Image
from src.data.masks import make_mask, get_mask_path
from src.data.labels import load_label, parse_buildings
from src.data.xbd import get_label_path
from src.data.metadata import build_metadata_table
from src.data.masks import save_masks
from src.data.split import make_split
from sklearn.model_selection import train_test_split

In [ ]:
ROOT = Path("/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd")

## 1. Mask generation
**comparing** the masks I generated with the pre-existing masks in the dataset

In [ ]:
FOLDER = "tier1"
image_id = "hurricane-harvey_00000000"

label = load_label(get_label_path(ROOT, FOLDER, image_id, "post"))

our_mask = make_mask(parse_buildings(label))

ref_path = ROOT / FOLDER / "masks" / f"{image_id}_post_disaster.png"
ref_mask = np.array(Image.open(ref_path))

print("Our mask values:      ", np.unique(our_mask))
print("Reference mask values:", np.unique(ref_mask))

same = (our_mask == ref_mask).mean()
print(f"Pixels that agree: {same:.2%}")

### test for all

In [ ]:
# metadata table
df = build_metadata_table(ROOT)
print(df.shape)

In [ ]:
with_unclassified = df[df["num_un-classified"] > 0].sample(50, random_state=42)
random_images = df.sample(50, random_state=42)
check = pd.concat([with_unclassified, random_images])

agreements = []
ref_values = set()

for _, row in check.iterrows():
    label = load_label(get_label_path(ROOT, row["folder"], row["image_id"], "post"))
    our_mask = make_mask(parse_buildings(label))

    ref_path = ROOT / row["folder"] / "masks" / f"{row['image_id']}_post_disaster.png"
    ref_mask = np.array(Image.open(ref_path))

    agreements.append((our_mask == ref_mask).mean())
    ref_values.update(np.unique(ref_mask).tolist())

agreements = np.array(agreements)
print("Images checked:", len(agreements))
print(f"Mean agreement: {agreements.mean():.4%}")
print(f"Min agreement:  {agreements.min():.4%}")
print("Values in reference masks:", sorted(ref_values))

In [ ]:
# un-classified mask check
ref_on_ignore = Counter()
agreements_wo_ignore = []

for _, row in check.iterrows():
    label = load_label(get_label_path(ROOT, row["folder"], row["image_id"], "post"))
    our_mask = make_mask(parse_buildings(label))
    ref_path = ROOT / row["folder"] / "masks" / f"{row['image_id']}_post_disaster.png"
    ref_mask = np.array(Image.open(ref_path))

    ignore = our_mask == 255
    values, counts = np.unique(ref_mask[ignore], return_counts=True)
    ref_on_ignore.update(dict(zip(values.tolist(), counts.tolist())))

    keep = ~ignore
    agreements_wo_ignore.append((our_mask[keep] == ref_mask[keep]).mean())

print("Reference values where our mask is 255:", dict(ref_on_ignore))
print(f"Mean agreement (without 255 pixels): {np.mean(agreements_wo_ignore):.4%}")
print(f"Min agreement  (without 255 pixels): {np.min(agreements_wo_ignore):.4%}")

**Findings**
**Mask validation**
- Our masks were compared with the pre-made masks of the Kaggle dataset on 100 images (50 random images and 50 images containing un-classified buildings).
- Both use the same encoding for damage classes (1–4).
- Excluding un-classified buildings, the masks agree on 99.996% of the pixels; the remaining differences are likely due to overlapping buildings.
- The pre-made masks draw un-classified buildings as background (0). Our masks mark them as 255, so that they are ignored in the loss instead of being learned as "no building". This confirms the decision to generate our own masks.

## 2. Saving Masks

In [15]:
MASK_ROOT = Path("/kaggle/working/masks")
save_masks(ROOT, df, MASK_ROOT)

saved = list(MASK_ROOT.rglob("*.png"))
print("Masks saved:", len(saved))

100%|██████████| 11034/11034 [01:46<00:00, 103.27it/s]


Masks saved: 11034
93M	/kaggle/working/masks


In [16]:
# example
mask = cv2.imread(str(get_mask_path(MASK_ROOT, "tier1", "hurricane-harvey_00000000")), cv2.IMREAD_UNCHANGED)
print(mask.shape, mask.dtype, np.unique(mask))

(1024, 1024) uint8 [0 2 3]


## 3. Stratified Split

In [21]:
train, validation, test = make_split(df)

print("Train:", len(train), f"({len(train) / len(df):.1%})")
print("Val:  ", len(validation), f"({len(validation) / len(df):.1%})")
print("Test: ", len(test), f"({len(test) / len(df):.1%})")

Train: 6620 (60.0%)
Val:   2207 (20.0%)
Test:  2207 (20.0%)


### 3.1 Leakage check
No pair should belong to both groups.

In [22]:
train_ids = set(train["image_id"])
val_ids = set(validation["image_id"])
test_ids = set(test["image_id"])

print("Train ∩ Val: ", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Val ∩ Test:  ", len(val_ids & test_ids))
print("Total unique:", len(train_ids | val_ids | test_ids))

Train ∩ Val:  0
Train ∩ Test: 0
Val ∩ Test:   0
Total unique: 11034


### 3.2 Stratified check
No pair should belong to both groups.

In [23]:
dist = pd.DataFrame({
    "all": df["disaster"].value_counts(normalize=True),
    "train": train["disaster"].value_counts(normalize=True),
    "val": validation["disaster"].value_counts(normalize=True),
    "test": test["disaster"].value_counts(normalize=True),
})

print((dist * 100).round(1))

                      all  train   val  test
disaster                                    
guatemala-volcano     0.3    0.3   0.2   0.3
hurricane-florence    4.9    4.9   5.0   4.9
hurricane-harvey      4.7    4.7   4.8   4.7
hurricane-matthew     3.7    3.7   3.7   3.7
hurricane-michael     5.0    5.0   5.0   5.0
joplin-tornado        1.4    1.3   1.4   1.4
lower-puna-volcano    2.6    2.6   2.6   2.6
mexico-earthquake     1.7    1.8   1.8   1.7
midwest-flooding      4.0    4.0   4.0   4.0
moore-tornado         2.1    2.1   2.0   2.1
nepal-flooding        5.6    5.6   5.6   5.6
palu-tsunami          1.8    1.8   1.8   1.8
pinery-bushfire      16.7   16.7  16.7  16.7
portugal-wildfire    16.9   16.9  16.9  16.9
santa-rosa-wildfire   3.4    3.4   3.4   3.4
socal-fire           12.7   12.7  12.7  12.7
sunda-tsunami         1.3    1.3   1.4   1.3
tuscaloosa-tornado    3.1    3.1   3.1   3.1
woolsey-fire          8.0    8.0   7.9   8.0


**Split checks**
- Sizes: 6,620 / 2,207 / 2,207 pairs (60% / 20% / 20%).
- No pair appears in more than one subset (no leakage), and all 11,034 pairs are assigned.
- The proportion of every disaster event is the same in train, val and test **(stratified by event)**.

### 3.3 Save the splits


In [24]:
SPLIT_DIR = Path("/kaggle/working/splits")
SPLIT_DIR.mkdir(exist_ok=True)

columns = ["image_id", "folder", "disaster", "disaster_type"]

for name, split_df in [("train", train), ("val", validation), ("test", test)]:
    out = split_df[columns].sort_values("image_id")
    out.to_csv(SPLIT_DIR / f"{name}.csv", index=False)
    print(name, len(out))

train 6620
val 2207
test 2207


In [26]:
# leakage test
!pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/damage-detection
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 3 items                                                              

tests/test_split_leakage.py::test_no_overlap PASSED                      [ 33%]
tests/test_split_leakage.py::test_all_assigned PASSED                    [ 66%]
tests/test_split_leakage.py::test_disaster_stratification PASSED         [100%]

============================== 3 passed in 0.62s ===============================
